# 00 — Environment check

Run this first on every machine (laptop and LinuxONE). The output doubles as **deployment evidence** for the judges:
architecture (`s390x` on LinuxONE), byte order (big-endian on IBM Z), library versions and CPU features.

In [ ]:
import importlib
import os
import platform
import sys
from pathlib import Path

print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {platform.platform()}")
print(f"Architecture : {platform.machine()}")
print(f"Byte order   : {sys.byteorder}")
print(f"CPU count    : {os.cpu_count()}")
print(f"On IBM Z     : {platform.machine() == 's390x'}")

Versions are compared against the official LinuxONE Community Cloud Jupyter Lab ML container
(`jupyterlab-image-s390x`). The scikit-learn version matters most: saved `.joblib` models only load reliably with the
version that wrote them.

In [ ]:
# name -> version in the official LinuxONE container (None = not pinned)
REFERENCE = {"numpy": None, "pandas": "3.0.6", "sklearn": "1.9.0", "xgboost": "3.4.1", "networkx": "3.7",
             "matplotlib": None, "joblib": None, "requests": None, "llama_cpp": "0.3.36"}

missing = []
for name, ref in REFERENCE.items():
    try:
        version = getattr(importlib.import_module(name), "__version__", "?")
    except ImportError:
        missing.append(name)
        print(f"{name:12s} MISSING")
        continue
    note = "" if ref is None else ("  (matches container)" if version == ref else f"  (container has {ref})")
    print(f"{name:12s} {version}{note}")

if missing:
    print("\nMissing:", ", ".join(missing), "— llama_cpp is optional; without it notebook 05 uses a template explanation.")

### Platform evidence for the judges
* **Container:** the official LinuxONE setup runs Jupyter inside a Docker container.
* **zDNN:** the container's llama.cpp is built with the zDNN backend, which runs Granite's matrix operations on the
  NNPA on-chip AI accelerator.

In [ ]:
in_container = (Path("/.dockerenv").exists() or Path("/run/.containerenv").exists()
                or any(k in Path("/proc/1/cgroup").read_text() for k in ("docker", "containerd", "libpod"))
                if Path("/proc/1/cgroup").exists() else False)
print("Running inside a container:", in_container)

try:
    import llama_cpp
    zdnn_libs = sorted((Path(llama_cpp.__file__).parent / "lib").glob("libggml-zdnn*.so"))
    print("llama.cpp zDNN backend   :", ", ".join(p.name for p in zdnn_libs) or "not found (CPU-only build)")
except ImportError:
    print("llama.cpp zDNN backend   : llama_cpp not installed")

On IBM Z, `/proc/cpuinfo` lists CPU facilities. `vx` is the vector facility; `nnpa` is the
Neural Network Processing Assist (the on-chip AI accelerator on z16 / LinuxONE 4 and newer).
Only claim accelerator use in the pitch if you actually route inference through it.

In [ ]:
cpuinfo = Path("/proc/cpuinfo")
if cpuinfo.exists():
    features = next((l.split(":", 1)[1].split() for l in cpuinfo.read_text().splitlines()
                     if l.startswith("features")), [])
    if features:
        print("CPU features:", " ".join(features))
        print("Vector facility (vx):", "vx" in features)
        print("AI accelerator (nnpa):", "nnpa" in features)
    else:
        print("No 'features' line (expected on x86/ARM; present on s390x).")